# 6.3 Games with chance: expectimax and Monte Carlo tree search: practice notebook

When dice or random events decide part of the game, plan for the average result. Back up expected values with expectimax and expectiminimax, see the problems the slides ask about, and choose moves from random games with Monte Carlo tree search.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Expectimax on the slide 11 tree

In [ ]:
# Expectimax on the example of the slides. MAX at the top, then chance nodes, then the outcome values.
tree = [[3, 12, 9], [2, 4, 6], [15, 6, 0]]
prob = [1/3, 1/3, 1/3]                 # every outcome of a chance node has probability 1/3

def exp_value(outcomes):
    v = 0
    for i in range(len(outcomes)):
        v = v + prob[i] * outcomes[i]  # v += p * value(successor)
    return v

values = []
for outcomes in tree:
    values.append(round(exp_value(outcomes), 2))
print("chance nodes:", values)
print("MAX picks", max(values), "with move", values.index(max(values)) + 1)

### Expectiminimax on slide 12 and the scaling problem

In [ ]:
# Expectiminimax on the middle branch of slide 12: a chance node over three MIN nodes.
min_nodes = [[5, 4, 2], [1, 7, 2], [3, 4, 5]]
prob = [0.5, 0.4, 0.1]

v = 0
for i in range(3):
    m = min(min_nodes[i])               # MIN picks its smallest value
    print("MIN node", i + 1, "=", m, " weight", prob[i])
    v = v + prob[i] * m
print("chance node =", round(v, 2))

# Why the size of the numbers matters: AIMA's example with chances 0.9 and 0.1.
for name, leaves in [("values 2 3 1 4", [2, 3, 1, 4]), ("values 20 30 1 400", [20, 30, 1, 400])]:
    move1 = 0.9 * leaves[0] + 0.1 * leaves[1]
    move2 = 0.9 * leaves[2] + 0.1 * leaves[3]
    print(name, ": move 1 =", round(move1, 2), ", move 2 =", round(move2, 2))

### UCB1 at the root

In [ ]:
# Monte Carlo search at the root with UCB1. Each move has a hidden chance of winning a random game.
import math, random
random.seed(2)
chance = [0.3, 0.6, 0.45]               # hidden from the search
wins = [0, 0, 0]
visits = [0, 0, 0]
c = 1.4                                 # how much to explore
for t in range(1, 21):
    pick = -1
    best = -1.0
    for i in range(3):
        if visits[i] == 0:
            score = 1000.0              # an untried move comes first
        else:
            score = wins[i] / visits[i] + c * math.sqrt(math.log(t - 1) / visits[i])
        if score > best:
            best = score
            pick = i
    result = 1 if random.random() < chance[pick] else 0     # play one random game
    wins[pick] = wins[pick] + result
    visits[pick] = visits[pick] + 1
print("wins  ", wins)
print("visits", visits)
print("play move", visits.index(max(visits)) + 1)

## Your turn

### Task 1: Expectimax with equal chances

Write `expectimax(node, is_max)`. A node is a number (a leaf) or a list of children. MAX nodes take the largest child value. The other nodes are chance nodes where every child is equally likely, so their value is the average of the children. Levels alternate, starting with MAX when is_max is True.

Hint: A leaf returns itself. Otherwise compute every child with the other kind of node, then take the max or the average.

In [ ]:
def expectimax(node, is_max):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('expectimax([[3, 12, 9], [2, 4, 6], [15, 6, 0]], True)', 8.0), ('expectimax([[1, 5], [4, 4]], True)', 4.0), ('expectimax([[[1, 3], [8, 2]], [[2, 2], [6, 0]]], False)', 4.0), ('expectimax(7, True)', 7)])

### Task 2: Expectiminimax with layer kinds

Write `expectiminimax(node, kinds, probs, depth)`. kinds is a list such as `["max", "chance", "min"]` that gives the node kind at each depth. probs is the list of probabilities used by every chance node, in child order. Return the value rounded to 2 decimals at depth 0 only.

Hint: Look up the kind with kinds[depth]. For a chance node add probs[i] times the value of child i.

In [ ]:
def expectiminimax(node, kinds, probs, depth):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("expectiminimax([[[5, 4, 2], [1, 7, 2], [3, 4, 5]]], ['max', 'chance', 'min'], [0.5, 0.4, 0.1], 0)", 1.7), ("expectiminimax([[3, 12, 9], [2, 4, 6], [15, 6, 0]], ['max', 'chance'], [0.2, 0.3, 0.5], 0)", 8.7), ("expectiminimax([[2, 3], [1, 4]], ['max', 'chance'], [0.9, 0.1], 0)", 2.1), ("expectiminimax([[20, 30], [1, 400]], ['max', 'chance'], [0.9, 0.1], 0)", 40.9)])

### Task 3: Pick a move with UCB1

Write `pick(wins, visits, c)`. wins and visits are lists with one entry per move. If some move has 0 visits, return the index of the first such move. Otherwise return the index of the move with the largest UCB1 = wins[i] ⁄ visits[i] + c × √(ln N ⁄ visits[i]), where N is the sum of the visits. On a tie, return the lowest index.

Hint: First look for an untried move. Then compute the UCB1 score of each move and keep the largest, using > so the lowest index wins a tie.

In [ ]:
import math

def pick(wins, visits, c):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('pick([0, 0, 0], [0, 0, 0], 1.4)', 0), ('pick([1, 0, 0], [1, 0, 0], 1.4)', 1), ('pick([3, 2, 1], [5, 3, 2], 0)', 1), ('pick([3, 2, 1], [5, 3, 2], 1.4)', 2), ('pick([6, 1], [10, 2], 0)', 0)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Expectimax with equal chances

Test 1 is the slide 11 tree. In test 3 the root is a chance node over two MAX nodes, and each MAX node is over two chance nodes. Left MAX: max(avg(1, 3), avg(8, 2)) = max(2, 5) = 5. Right MAX: max(avg(2, 2), avg(6, 0)) = max(2, 3) = 3. Root: avg(5, 3) = 4.

In [ ]:
def expectimax(node, is_max):
    if type(node) == int:
        return node
    values = []
    for child in node:
        values.append(expectimax(child, not is_max))
    if is_max:
        return max(values)
    return sum(values) / len(values)

In [ ]:
check([('expectimax([[3, 12, 9], [2, 4, 6], [15, 6, 0]], True)', 8.0), ('expectimax([[1, 5], [4, 4]], True)', 4.0), ('expectimax([[[1, 3], [8, 2]], [[2, 2], [6, 0]]], False)', 4.0), ('expectimax(7, True)', 7)])

### Solution to task 2: Expectiminimax with layer kinds

Test 1 is the slide 12 branch. Test 2 uses other chances on the slide 11 tree: 0.2 × 3 + 0.3 × 12 + 0.5 × 9 = 8.7, and the other two chance nodes give 4.6 and 4.8, so the root is max(8.7, 4.6, 4.8) = 8.7. Tests 3 and 4 are the AIMA scaling example.

In [ ]:
def expectiminimax(node, kinds, probs, depth):
    if type(node) == int or type(node) == float:
        return node
    values = []
    for child in node:
        values.append(expectiminimax(child, kinds, probs, depth + 1))
    kind = kinds[depth]
    if kind == "max":
        v = max(values)
    elif kind == "min":
        v = min(values)
    else:
        v = 0
        for i in range(len(values)):
            v = v + probs[i] * values[i]
    if depth == 0:
        return round(v, 2)
    return v

In [ ]:
check([("expectiminimax([[[5, 4, 2], [1, 7, 2], [3, 4, 5]]], ['max', 'chance', 'min'], [0.5, 0.4, 0.1], 0)", 1.7), ("expectiminimax([[3, 12, 9], [2, 4, 6], [15, 6, 0]], ['max', 'chance'], [0.2, 0.3, 0.5], 0)", 8.7), ("expectiminimax([[2, 3], [1, 4]], ['max', 'chance'], [0.9, 0.1], 0)", 2.1), ("expectiminimax([[20, 30], [1, 400]], ['max', 'chance'], [0.9, 0.1], 0)", 40.9)])

### Solution to task 3: Pick a move with UCB1

Test 3 has c = 0, so only the win rate counts: 0.6, 0.67, 0.5. Test 4 adds exploration: the scores are about 1.55, 1.89 and 2.00, so the move with only 2 visits wins.

In [ ]:
import math

def pick(wins, visits, c):
    for i in range(len(visits)):
        if visits[i] == 0:
            return i
    total = sum(visits)
    best = 0
    best_score = -1.0
    for i in range(len(visits)):
        score = wins[i] / visits[i] + c * math.sqrt(math.log(total) / visits[i])
        if score > best_score:
            best_score = score
            best = i
    return best

In [ ]:
check([('pick([0, 0, 0], [0, 0, 0], 1.4)', 0), ('pick([1, 0, 0], [1, 0, 0], 1.4)', 1), ('pick([3, 2, 1], [5, 3, 2], 0)', 1), ('pick([3, 2, 1], [5, 3, 2], 1.4)', 2), ('pick([6, 1], [10, 2], 0)', 0)])